# tndrr assignment - Colab runner
Runtime -> Change runtime type -> **T4 GPU**. Run the cells top to bottom. Upload/clone this repo first.

In [ ]:
!nvidia-smi | head -15
# Option A: clone your GitHub copy:  !git clone https://github.com/<you>/<repo>.git repo
# Option B: upload the zip and unzip it:
# from google.colab import files; files.upload(); !unzip -q tndrr-assignment.zip
%cd /content/tndrr-assignment

## Task 1 - LoRA

In [ ]:
%cd task1_lora
!pip install -q -r requirements.txt
!bash run_all.sh

In [ ]:
!bash ablation.sh   # optional

In [ ]:
!python memory_calc.py --name Qwen2.5-0.5B --params 0.494e9 --layers 24 --hidden 896 --inter 4864 --heads 14 --kv-heads 2 --act-gb 1 --out results/memory_calc_0p5b.json
!cat outputs/lora_r16/train_summary.json | head -40
%cd ..

## Task 2 - RAG
Write `task2_rag/questions.json` (15 questions, YOUR OWN) before running the evaluation. Use `find_passage.py` to copy gold passages.

In [ ]:
%cd task2_rag
!pip install -q -r requirements.txt
!python fetch_docs.py
!python find_passage.py "low-rank" --docs docs --max 3

In [ ]:
# after you created questions.json:
!python evaluate_retrieval.py --chunk-sizes 100 250 --rerank
!cat results/retrieval_failures.md | head -60
%cd ..

## Task 3 - Ollama (16-bit vs 4-bit)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT); time.sleep(8)
!ollama pull qwen2.5:1.5b-instruct-fp16
!ollama pull qwen2.5:1.5b-instruct-q4_K_M
!ollama list

In [ ]:
%cd task3_local_inference
!python benchmark_ollama.py --models qwen2.5:1.5b-instruct-fp16 qwen2.5:1.5b-instruct-q4_K_M
!python compare_quality.py
!python kv_cache_calc.py --ctx 4096 --users 20
!python serving_plan.py --users 20 --ctx 4096
%cd ..

## Report
Fill `report/my_answers.md` (own words), then:

In [ ]:
!pip install -q reportlab pypdf
!python report/build_report.py --name "YOUR NAME" --github https://github.com/<you>/<repo> --hardware "Colab T4 16GB"
# download results + report:  from google.colab import files; files.download('report/report.pdf')